In [ ]:
!pip install roboflow

In [ ]:
from roboflow import Roboflow

# Ambil API key dari Colab Secrets (ikon kunci di sidebar kiri).
# Nama secret: ROBOFLOW_API_KEY. Kalau dijalankan di luar Colab atau
# secret belum diset, fallback ke input manual.
try:
    from google.colab import userdata
    api_key = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    from getpass import getpass
    api_key = getpass("Masukkan Roboflow API Key Anda: ")

rf = Roboflow(api_key=api_key)
project = rf.workspace("joseph-nelson").project("mask-wearing")
version = project.version(1)
dataset = version.download("yolov8")

data_path = dataset.location + '/data.yaml'  # Otomatis mengatur path file YAML

In [ ]:
# Pastikan YOLOv8 sudah diinstal
!pip install ultralytics

# Memastikan GPU tersedia
!nvidia-smi

In [ ]:
from ultralytics import YOLO
import os

# Load model YOLOv8
model = YOLO('yolov8n.pt')  # Menggunakan model YOLOv8 Nano

# Pastikan file data.yaml berada di lokasi yang benar
data_path = '/content/Mask-Wearing-1/data.yaml'  # Sesuaikan path ini jika data diunggah atau diakses dari Google Drive

# Melatih model
model.train(
    data=data_path,  # Path ke file data.yaml
    epochs=500,  # Jumlah epoch
    imgsz=640,  # Ukuran gambar
    batch=16,   # Batch size (disarankan kecil jika memori GPU terbatas, misalnya 16)
    workers=2,  # Jumlah workers untuk dataloader
    name='yolov8n_experiment',  # Nama eksperimen
    device='0'  # Menggunakan GPU pertama
)

# Evaluasi performa model pada validation set
metrics = model.val()

In [ ]:
# ### Simpan Model ke Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil

drive_save_dir = '/content/drive/MyDrive/yolo-mask-detection'
os.makedirs(drive_save_dir, exist_ok=True)

best_model_path = 'runs/detect/yolov8n_experiment/weights/best.pt'
shutil.copy(best_model_path, f'{drive_save_dir}/best.pt')
print(f"Model disimpan ke {drive_save_dir}/best.pt")

## Eksperimen v2: Model Lebih Besar + Resolusi Training Lebih Tinggi

Model v1 (`yolov8n`, imgsz=640, mAP50=0.849 di validation) ternyata masih
melewatkan beberapa wajah saat diuji di foto CCTV nyata (lihat catatan
keterbatasan di `Test_Model_Yolo_Mask.ipynb`). Dugaan utamanya: kapasitas
model nano dan resolusi training 640 kurang cukup untuk wajah kecil/jauh.

v2 ini menguji dugaan tersebut dengan mengganti dua hal sekaligus:
- `yolov8n.pt` -> `yolov8s.pt` (kapasitas lebih besar)
- `imgsz=640` -> `imgsz=960` (lebih banyak detail untuk objek kecil)

Dataset dan jumlah epoch tetap sama seperti v1, supaya perbandingannya adil.

In [ ]:
# ### Training v2: yolov8s, imgsz=960
model_v2 = YOLO('yolov8s.pt')

model_v2.train(
    data=data_path,       # dataset sama seperti v1 (dari cell download di atas)
    epochs=500,
    imgsz=960,             # naik dari 640 -- lebih banyak detail utk wajah kecil/jauh
    batch=16,
    workers=2,
    name='yolov8s_experiment',
    device='0'
)

metrics_v2 = model_v2.val()

In [ ]:
# ### Simpan Model v2 ke Google Drive (nama file berbeda dari v1)
from google.colab import drive
drive.mount('/content/drive')

import glob

drive_save_dir = '/content/drive/MyDrive/yolo-mask-detection'
os.makedirs(drive_save_dir, exist_ok=True)

# Dicari lewat glob (bukan hardcode nama folder) -- kalau cell training
# di atas sempat dijalankan lebih dari sekali, Ultralytics otomatis
# menambah angka di belakang nama run (mis. 'yolov8s_experiment2'), jadi path
# tetap ketemu meski nama foldernya berubah.
candidates = sorted(
    glob.glob('runs/detect/yolov8s_experiment*/weights/best.pt'),
    key=os.path.getmtime
)
if not candidates:
    raise FileNotFoundError(
        "best.pt tidak ditemukan -- cek apakah cell training di atas benar-benar "
        "selesai tanpa error (scroll ke atas, lihat output training-nya)."
    )
best_model_path = candidates[-1]  # paling baru kalau ada lebih dari satu
print(f"Menggunakan checkpoint: {best_model_path}")

shutil.copy(best_model_path, f'{drive_save_dir}/best_v2.pt')
print(f"Model disimpan ke {drive_save_dir}/best_v2.pt")

## Eksperimen v3: Tambah Data Training (Lebih Beragam)

v1 dan v2 sama-sama dilatih di dataset yang sama (105 gambar train).
Pengujian di foto CCTV nyata menunjukkan beberapa kasus (terutama
kombinasi hijab+masker tertentu) tidak pernah terdeteksi di kedua versi,
di semua kombinasi conf/imgsz/nms/TTA yang sudah dicoba -- indikasi kuat
ini soal keterbatasan data, bukan parameter.

v3 menguji itu dengan menambah data dari dataset publik lain
("Face Mask Detection" oleh Hyssam Baccouche, 848 gambar, mirror dari
dataset akademik Kaggle andrewmvd/face-mask-detection -- lebih banyak
foto dengan beberapa orang sekaligus per gambar). Dataset ini pakai 3
kelas (with_mask, without_mask, mask_weared_incorrect), jadi perlu
dipetakan dulu ke skema kita (mask, no-mask) sebelum digabung.

Catatan pemetaan: `mask_weared_incorrect` saya petakan ke `no-mask`
(masker dipakai tidak benar = tidak terlindungi secara efektif) --
ini keputusan yang bisa didiskusikan, ubah CLASS_MAP di bawah kalau
mau perlakuan berbeda.

In [ ]:
# ### Download Dataset Tambahan
extra_project = rf.workspace("hyssam-baccouche").project("face-mask-detection-djzcc")
extra_version = extra_project.version(1)
extra_dataset = extra_version.download("yolov8")

In [ ]:
# ### Petakan Kelas & Gabungkan ke Dataset Utama
import yaml
import shutil
from pathlib import Path

# Baca nama kelas kedua dataset langsung dari data.yaml masing-masing
# (jangan asumsikan urutan index -- selalu beda-beda tergantung export)
with open(f"{extra_dataset.location}/data.yaml") as f:
    extra_names = yaml.safe_load(f)['names']
with open(f"{dataset.location}/data.yaml") as f:
    base_names = yaml.safe_load(f)['names']

print("Kelas dataset utama:", base_names)
print("Kelas dataset tambahan:", extra_names)

CLASS_MAP = {
    "with_mask": "mask",
    "without_mask": "no-mask",
    "mask_weared_incorrect": "no-mask",
}

name_to_base_idx = {name: i for i, name in enumerate(base_names)}
extra_idx_to_base_idx = {
    i: name_to_base_idx[CLASS_MAP[name]]
    for i, name in enumerate(extra_names)
    if name in CLASS_MAP
}
print("Pemetaan index (extra -> utama):", extra_idx_to_base_idx)


def merge_split(split):
    src_img_dir = Path(extra_dataset.location) / split / "images"
    src_lbl_dir = Path(extra_dataset.location) / split / "labels"
    dst_img_dir = Path(dataset.location) / split / "images"
    dst_lbl_dir = Path(dataset.location) / split / "labels"

    if not src_img_dir.exists():
        print(f"Split '{split}' tidak ada di dataset tambahan, dilewati.")
        return 0

    count = 0
    for img_path in src_img_dir.glob("*"):
        lbl_path = src_lbl_dir / (img_path.stem + ".txt")
        if not lbl_path.exists():
            continue

        new_lines = []
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                old_idx = int(parts[0])
                if old_idx not in extra_idx_to_base_idx:
                    continue
                new_idx = extra_idx_to_base_idx[old_idx]
                new_lines.append(" ".join([str(new_idx)] + parts[1:]))

        if not new_lines:
            continue

        new_name = f"extra_{img_path.name}"
        shutil.copy(img_path, dst_img_dir / new_name)
        with open(dst_lbl_dir / (Path(new_name).stem + ".txt"), "w") as f:
            f.write("\n".join(new_lines))
        count += 1

    print(f"{split}: {count} gambar ditambahkan dari dataset tambahan")
    return count


n_train_added = merge_split("train")
n_valid_added = merge_split("valid")
print(f"\nTotal gambar tambahan: {n_train_added + n_valid_added}")

In [ ]:
# ### Training v3: yolov8s, imgsz=960, dataset gabungan
# Sama seperti v2 (model & resolusi training sama) -- satu-satunya variabel
# yang berubah adalah jumlah & keragaman data, supaya perbandingan v2 vs v3
# adil (isolasi efek 'data lebih banyak' saja).
model_v3 = YOLO('yolov8s.pt')

model_v3.train(
    data=data_path,
    epochs=500,
    imgsz=960,
    batch=16,
    workers=2,
    name='yolov8s_v3_experiment',
    device='0'
)

metrics_v3 = model_v3.val()

In [ ]:
# ### Simpan Model v3 ke Google Drive
from google.colab import drive
drive.mount('/content/drive')

import glob

drive_save_dir = '/content/drive/MyDrive/yolo-mask-detection'
os.makedirs(drive_save_dir, exist_ok=True)

# Dicari lewat glob (bukan hardcode nama folder) -- kalau cell training
# di atas sempat dijalankan lebih dari sekali, Ultralytics otomatis
# menambah angka di belakang nama run (mis. 'yolov8s_v3_experiment2'), jadi path
# tetap ketemu meski nama foldernya berubah.
candidates = sorted(
    glob.glob('runs/detect/yolov8s_v3_experiment*/weights/best.pt'),
    key=os.path.getmtime
)
if not candidates:
    raise FileNotFoundError(
        "best.pt tidak ditemukan -- cek apakah cell training di atas benar-benar "
        "selesai tanpa error (scroll ke atas, lihat output training-nya)."
    )
best_model_path = candidates[-1]  # paling baru kalau ada lebih dari satu
print(f"Menggunakan checkpoint: {best_model_path}")

shutil.copy(best_model_path, f'{drive_save_dir}/best_v3.pt')
print(f"Model disimpan ke {drive_save_dir}/best_v3.pt")

## Eksperimen v4: Rebalance Kelas Dataset Tambahan

Pengujian nyata menunjukkan v3 (data gabungan) jadi bias memprediksi
"mask" -- termasuk pada wajah yang jelas TIDAK pakai masker (v1/v2 benar,
v3 salah pada foto yang sama). Dugaan: dataset tambahan (848 gambar,
sumber Kaggle andrewmvd) kelasnya tidak seimbang -- `with_mask` biasanya
jauh lebih dominan daripada `without_mask` + `mask_weared_incorrect`.

v4 ini: (1) mengukur imbalance-nya secara langsung dari data yang sudah
diunduh (bukan menebak), (2) membuang sebagian gambar tambahan yang
HANYA berisi kelas mask (tidak menyentuh gambar yang punya contoh
no-mask, supaya keragaman yang sudah didapat v3 tidak hilang), sampai
rasio mask:no-mask di data tambahan mendekati seimbang, lalu (3) retrain
dengan konfigurasi sama seperti v3 (yolov8s, imgsz=960).

In [ ]:
# ### Cek Keseimbangan Kelas
from collections import Counter
from pathlib import Path


def count_class_instances(label_dir, class_names, only_extra=None):
    """Hitung jumlah instance per kelas di semua file label pada folder.
    only_extra=True  -> cuma file 'extra_*.txt' (data tambahan)
    only_extra=False -> cuma file BUKAN 'extra_*.txt' (data asli)
    only_extra=None  -> semua file
    """
    counts = Counter()
    for lbl_file in Path(label_dir).glob("*.txt"):
        is_extra = lbl_file.name.startswith("extra_")
        if only_extra is True and not is_extra:
            continue
        if only_extra is False and is_extra:
            continue
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                counts[class_names[int(parts[0])]] += 1
    return counts


# Distribusi ASLI dataset tambahan (sebelum remap ke skema kita -- 3 kelas asli)
extra_raw_counts = Counter()
for lbl_file in Path(extra_dataset.location, "train", "labels").glob("*.txt"):
    with open(lbl_file) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            extra_raw_counts[extra_names[int(parts[0])]] += 1
print("Distribusi kelas ASLI dataset tambahan (sebelum remap):", dict(extra_raw_counts))

train_label_dir = f"{dataset.location}/train/labels"
original_counts = count_class_instances(train_label_dir, base_names, only_extra=False)
extra_counts = count_class_instances(train_label_dir, base_names, only_extra=True)

print("\nDistribusi kelas data ASLI (105 gambar):", dict(original_counts))
print("Distribusi kelas data TAMBAHAN setelah remap (763 gambar):", dict(extra_counts))
ratio = extra_counts['mask'] / max(extra_counts['no-mask'], 1)
print(f"\nRasio mask:no-mask di data tambahan -> {extra_counts['mask']}:{extra_counts['no-mask']} (~{ratio:.1f}x lebih banyak mask)")

In [ ]:
# ### Rebalance: Buang Sebagian Gambar Tambahan yang Hanya Berisi Kelas Mask
import random
random.seed(42)


def get_image_classes(lbl_path, class_names):
    classes = set()
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            classes.add(class_names[int(parts[0])])
    return classes


train_img_dir = Path(dataset.location) / "train" / "images"
train_lbl_dir = Path(dataset.location) / "train" / "labels"

# Cari gambar tambahan yang HANYA berisi kelas mask (tidak ada contoh no-mask
# di gambar itu) -- ini yang jadi kandidat dibuang, gambar yang punya contoh
# no-mask (walau juga ada mask di dalamnya) tidak disentuh sama sekali
mask_only_extra = [
    p for p in train_lbl_dir.glob("extra_*.txt")
    if get_image_classes(p, base_names) == {"mask"}
]
print(f"Gambar tambahan yang hanya berisi kelas 'mask': {len(mask_only_extra)}")

excess_mask = extra_counts['mask'] - extra_counts['no-mask']
n_to_remove = max(0, min(len(mask_only_extra), excess_mask))

random.shuffle(mask_only_extra)
to_remove = mask_only_extra[:n_to_remove]

for lbl_path in to_remove:
    img_stem = lbl_path.stem
    lbl_path.unlink()
    for img_path in train_img_dir.glob(f"{img_stem}.*"):
        img_path.unlink()

print(f"Dihapus {len(to_remove)} gambar (hanya kelas mask) untuk menyeimbangkan data")

new_extra_counts = count_class_instances(str(train_lbl_dir), base_names, only_extra=True)
print("Distribusi kelas data tambahan setelah rebalance:", dict(new_extra_counts))

In [ ]:
# ### Training v4: yolov8s, imgsz=960, dataset gabungan (di-rebalance)
model_v4 = YOLO('yolov8s.pt')

model_v4.train(
    data=data_path,
    epochs=500,
    imgsz=960,
    batch=16,
    workers=2,
    name='yolov8s_v4_experiment',
    device='0'
)

metrics_v4 = model_v4.val()

In [ ]:
# ### Simpan Model v4 ke Google Drive
from google.colab import drive
drive.mount('/content/drive')

import glob

drive_save_dir = '/content/drive/MyDrive/yolo-mask-detection'
os.makedirs(drive_save_dir, exist_ok=True)

# Dicari lewat glob (bukan hardcode nama folder) -- kalau cell training
# di atas sempat dijalankan lebih dari sekali, Ultralytics otomatis
# menambah angka di belakang nama run (mis. 'yolov8s_v4_experiment2'), jadi path
# tetap ketemu meski nama foldernya berubah.
candidates = sorted(
    glob.glob('runs/detect/yolov8s_v4_experiment*/weights/best.pt'),
    key=os.path.getmtime
)
if not candidates:
    raise FileNotFoundError(
        "best.pt tidak ditemukan -- cek apakah cell training di atas benar-benar "
        "selesai tanpa error (scroll ke atas, lihat output training-nya)."
    )
best_model_path = candidates[-1]  # paling baru kalau ada lebih dari satu
print(f"Menggunakan checkpoint: {best_model_path}")

shutil.copy(best_model_path, f'{drive_save_dir}/best_v4.pt')
print(f"Model disimpan ke {drive_save_dir}/best_v4.pt")

## Eksperimen v5: Cek & Rebalance Keseimbangan Kelas TOTAL

v4 memperbaiki SEBAGIAN bias ke "mask" (dengan merebalance 763 gambar
tambahan), tapi bias residual masih muncul di sejumlah foto uji baru
(wajah tanpa masker sesekali masih salah terklasifikasi). Dugaan: v4
cuma menyeimbangkan porsi data TAMBAHAN -- keseimbangan kelas di 105
gambar dataset ASLI, dan distribusi TOTAL gabungan keduanya, belum pernah
dicek.

v5 ini: (1) hitung distribusi kelas TOTAL yang sekarang ada di folder
dataset (105 asli + data tambahan yang sudah di-rebalance v4), (2) kalau
masih timpang, buang lebih banyak gambar "mask-only" -- diprioritaskan
dari data TAMBAHAN dulu, baru dari data ASLI kalau memang belum cukup,
supaya 105 gambar asli sebisa mungkin tetap utuh -- sampai rasionya
mendekati seimbang, lalu (3) retrain.

### Jalan Pintas: Sudah Punya .pt v1-v4, Tidak Mau Training Ulang

Kalau v1-v4 sudah pernah dilatih dan file `.pt`-nya sudah aman di Drive,
**tidak perlu** menjalankan cell 0-17 di atas satu per satu (itu akan
training ulang v1-v4 dari nol, lama). Cukup jalankan SATU cell di bawah
ini -- isinya cuma mengulang bagian download + gabung + rebalance v4
(tanpa training apa pun) untuk menyiapkan ulang variabel & folder
dataset yang dibutuhkan cell v5 (variabel-variabel itu cuma ada selama
satu sesi Colab, hilang kalau runtime di-restart -- beda dengan `.pt`
yang sudah aman tersimpan di Drive). Setelah cell ini selesai, langsung
lanjut ke "Cek Keseimbangan Kelas TOTAL" di bawah, skip semuanya di atas.

In [ ]:
# ### Fast-forward: Siapkan Ulang Dataset Tanpa Training v1-v4
!pip install roboflow ultralytics -q

import os
import shutil
import random
import yaml
from pathlib import Path
from collections import Counter
from getpass import getpass
from ultralytics import YOLO
from roboflow import Roboflow

# 1. Download dataset dasar (105 gambar) -- sama seperti cell 1
try:
    from google.colab import userdata
    api_key = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    api_key = getpass("Masukkan Roboflow API Key Anda: ")

rf = Roboflow(api_key=api_key)
project = rf.workspace("joseph-nelson").project("mask-wearing")
version = project.version(1)
dataset = version.download("yolov8")
data_path = dataset.location + '/data.yaml'

# 2. Download dataset tambahan (848 gambar) -- sama seperti cell 9
extra_project = rf.workspace("hyssam-baccouche").project("face-mask-detection-djzcc")
extra_version = extra_project.version(1)
extra_dataset = extra_version.download("yolov8")

# 3. Petakan kelas & gabungkan -- sama seperti cell 10
with open(f"{extra_dataset.location}/data.yaml") as f:
    extra_names = yaml.safe_load(f)['names']
with open(f"{dataset.location}/data.yaml") as f:
    base_names = yaml.safe_load(f)['names']
print("Kelas dataset utama:", base_names)
print("Kelas dataset tambahan:", extra_names)

CLASS_MAP = {
    "with_mask": "mask",
    "without_mask": "no-mask",
    "mask_weared_incorrect": "no-mask",
}
name_to_base_idx = {name: i for i, name in enumerate(base_names)}
extra_idx_to_base_idx = {
    i: name_to_base_idx[CLASS_MAP[name]]
    for i, name in enumerate(extra_names)
    if name in CLASS_MAP
}


def merge_split(split):
    src_img_dir = Path(extra_dataset.location) / split / "images"
    src_lbl_dir = Path(extra_dataset.location) / split / "labels"
    dst_img_dir = Path(dataset.location) / split / "images"
    dst_lbl_dir = Path(dataset.location) / split / "labels"
    if not src_img_dir.exists():
        return 0
    count = 0
    for img_path in src_img_dir.glob("*"):
        lbl_path = src_lbl_dir / (img_path.stem + ".txt")
        if not lbl_path.exists():
            continue
        new_lines = []
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                old_idx = int(parts[0])
                if old_idx not in extra_idx_to_base_idx:
                    continue
                new_idx = extra_idx_to_base_idx[old_idx]
                new_lines.append(" ".join([str(new_idx)] + parts[1:]))
        if not new_lines:
            continue
        new_name = f"extra_{img_path.name}"
        shutil.copy(img_path, dst_img_dir / new_name)
        with open(dst_lbl_dir / (Path(new_name).stem + ".txt"), "w") as f:
            f.write("\n".join(new_lines))
        count += 1
    return count


n_train_added = merge_split("train")
n_valid_added = merge_split("valid")
print(f"Gambar tambahan digabung: {n_train_added + n_valid_added}")


# 4. Rebalance v4 -- sama seperti cell 14+15 (seed sama, hasil konsisten dengan v4 asli)
def count_class_instances(label_dir, class_names, only_extra=None):
    counts = Counter()
    for lbl_file in Path(label_dir).glob("*.txt"):
        is_extra = lbl_file.name.startswith("extra_")
        if only_extra is True and not is_extra:
            continue
        if only_extra is False and is_extra:
            continue
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                counts[class_names[int(parts[0])]] += 1
    return counts


def get_image_classes(lbl_path, class_names):
    classes = set()
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            classes.add(class_names[int(parts[0])])
    return classes


train_img_dir = Path(dataset.location) / "train" / "images"
train_lbl_dir = Path(dataset.location) / "train" / "labels"

random.seed(42)
extra_counts = count_class_instances(str(train_lbl_dir), base_names, only_extra=True)
mask_only_extra = [
    p for p in train_lbl_dir.glob("extra_*.txt")
    if get_image_classes(p, base_names) == {"mask"}
]
random.shuffle(mask_only_extra)
excess_mask = extra_counts['mask'] - extra_counts['no-mask']
n_to_remove = max(0, min(len(mask_only_extra), excess_mask))
for lbl_path in mask_only_extra[:n_to_remove]:
    img_stem = lbl_path.stem
    lbl_path.unlink()
    for img_path in train_img_dir.glob(f"{img_stem}.*"):
        img_path.unlink()
print(f"Rebalance v4: dihapus {n_to_remove} gambar mask-only dari data tambahan")

print("\nDataset siap (setara kondisi setelah cell 0-17 dijalankan, tanpa training).")
print("Distribusi saat ini:", dict(count_class_instances(str(train_lbl_dir), base_names, only_extra=None)))
print("Lanjut ke 'Cek Keseimbangan Kelas TOTAL' di bawah.")

In [ ]:
# ### Cek Keseimbangan Kelas TOTAL (Asli + Tambahan Hasil Rebalance v4)
train_label_dir = f"{dataset.location}/train/labels"

total_counts = count_class_instances(train_label_dir, base_names, only_extra=None)
original_counts_now = count_class_instances(train_label_dir, base_names, only_extra=False)
extra_counts_now = count_class_instances(train_label_dir, base_names, only_extra=True)

print("Distribusi kelas data ASLI (105 gambar):", dict(original_counts_now))
print("Distribusi kelas data TAMBAHAN (setelah rebalance v4):", dict(extra_counts_now))
print("Distribusi kelas TOTAL (gabungan):", dict(total_counts))

ratio_total = total_counts['mask'] / max(total_counts['no-mask'], 1)
print(f"\nRasio TOTAL mask:no-mask -> {total_counts['mask']}:{total_counts['no-mask']} (~{ratio_total:.2f}x lebih banyak mask)")

In [ ]:
# ### Rebalance Tambahan: Buang Lebih Banyak Gambar 'Mask-Only' Sampai Total Seimbang
import random
random.seed(43)  # beda dari seed v4 supaya tidak menyasar gambar yang persis sama

TARGET_RATIO = 1.5  # target rasio mask:no-mask yang dianggap cukup seimbang

train_img_dir = Path(dataset.location) / "train" / "images"
train_lbl_dir = Path(dataset.location) / "train" / "labels"


def instance_counts_in_file(lbl_path, class_names):
    counts = Counter()
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            counts[class_names[int(parts[0])]] += 1
    return counts


def is_mask_only(lbl_path, class_names):
    counts = instance_counts_in_file(lbl_path, class_names)
    return set(counts.keys()) == {"mask"}


# Prioritaskan buang dari data TAMBAHAN dulu, baru dari data ASLI kalau
# memang belum cukup -- supaya 105 gambar asli sebisa mungkin tetap utuh
extra_mask_only = [p for p in train_lbl_dir.glob("extra_*.txt") if is_mask_only(p, base_names)]
original_mask_only = [
    p for p in train_lbl_dir.glob("*.txt")
    if not p.name.startswith("extra_") and is_mask_only(p, base_names)
]
random.shuffle(extra_mask_only)
random.shuffle(original_mask_only)

print(f"Kandidat mask-only -- tambahan: {len(extra_mask_only)}, asli: {len(original_mask_only)}")

current_mask = total_counts['mask']
current_nomask = total_counts['no-mask']
removed = []

for pool_name, pool in [("tambahan", extra_mask_only), ("asli", original_mask_only)]:
    for lbl_path in pool:
        if current_mask / max(current_nomask, 1) <= TARGET_RATIO:
            break
        n_mask_in_file = instance_counts_in_file(lbl_path, base_names).get("mask", 0)
        current_mask -= n_mask_in_file
        removed.append((pool_name, lbl_path))
    if current_mask / max(current_nomask, 1) <= TARGET_RATIO:
        break

for _, lbl_path in removed:
    img_stem = lbl_path.stem
    lbl_path.unlink()
    for img_path in train_img_dir.glob(f"{img_stem}.*"):
        img_path.unlink()

n_removed_extra = sum(1 for p, _ in removed if p == "tambahan")
n_removed_original = sum(1 for p, _ in removed if p == "asli")
print(f"Dihapus {len(removed)} gambar mask-only "
      f"({n_removed_extra} dari data tambahan, {n_removed_original} dari data asli)")
if n_removed_original > 0:
    print(f"PERHATIAN: {n_removed_original} gambar dari 105 data ASLI ikut dibuang "
          f"-- data tambahan saja tidak cukup untuk menyeimbangkan rasio target.")

new_total_counts = count_class_instances(str(train_lbl_dir), base_names, only_extra=None)
new_ratio = new_total_counts['mask'] / max(new_total_counts['no-mask'], 1)
print(f"Distribusi kelas TOTAL setelah rebalance v5: {dict(new_total_counts)} (rasio ~{new_ratio:.2f}x)")

In [ ]:
# ### Training v5: yolov8s, imgsz=960, dataset gabungan (di-rebalance total)
model_v5 = YOLO('yolov8s.pt')

model_v5.train(
    data=data_path,
    epochs=500,
    imgsz=960,
    batch=16,
    workers=2,
    name='yolov8s_v5_experiment',
    device='0'
)

metrics_v5 = model_v5.val()

In [ ]:
# ### Simpan Model v5 ke Google Drive
from google.colab import drive
drive.mount('/content/drive')

import glob

drive_save_dir = '/content/drive/MyDrive/yolo-mask-detection'
os.makedirs(drive_save_dir, exist_ok=True)

candidates = sorted(
    glob.glob('runs/detect/yolov8s_v5_experiment*/weights/best.pt'),
    key=os.path.getmtime
)
if not candidates:
    raise FileNotFoundError(
        "best.pt tidak ditemukan -- cek apakah cell training di atas benar-benar "
        "selesai tanpa error (scroll ke atas, lihat output training-nya)."
    )
best_model_path = candidates[-1]
print(f"Menggunakan checkpoint: {best_model_path}")
shutil.copy(best_model_path, f'{drive_save_dir}/best_v5.pt')
print(f"Model disimpan ke {drive_save_dir}/best_v5.pt")

---
*Case study developed as part of the rubythalib.ai AI Engineer Bootcamp (CV Super Class, Object Detection module).*